In [69]:
mutable struct SimulationData
    # contains a vector of dictionaries with input parameters and their names
    # contains a vector of outputs for each of the input dictionaries
    input_variables::Vector{Dict{String,Any}}
    simulation_Results::Vector{Dict{String,Any}}

    function SimulationData()
        input_variables = []
        simulation_Results = []
        new(input_variables, simulation_Results)
    end
end

In [139]:
using JLD2
using Dates

function save_io(func::Function, args...; printing::Bool=false, redo::Bool=false, kwargs...)
    #Saves the inputs and outputs of a function to a file.
    # Generate filename from function name
    # file contains "input" and "output" variables
    # input is a vector of dictionaries containing the input arguments
    # output is a vector of output arguments in arbitrary format
    run_func::Bool = true
    check_input::Bool = false
    filename = joinpath("Results", string(func)*".jld2")
    if !isdir("Results")
        mkdir("Results")
        run_func = true
    end
    # does the file exist?
    exists::Bool = isfile(filename)
    # if it does, load it using JLD2
    if exists
        # check 
        input = nothing
        jldopen(filename, "r") do file
            # check if input exists in file
            all_keys = keys(file)
            if "input" in all_keys
                # check if args and kwargs exist in file
                check_input = true
                input = file["input"]
            end
        end
    end
    # generate tags for input vector names
    # make a copy of the input arguments in case they get changed by func
    args = deepcopy(args)
    kwargs = deepcopy(kwargs)
    # find method of func that will be dispatched with args (kwargs doesn't pplay a role for this)
    # Use `which` to find the corresponding method
    dispatched_method = which(func, Tuple(typeof.(args)))
    argument_names = Base.method_argnames(dispatched_method)[2:end]
    input_variables::Dict{String, Any} = Dict()
    for (arg_name, arg) in zip(argument_names, args)
        input_variables[string(arg_name)] = arg
    end
    for (key, value) in kwargs
        input_variables[string(key)] = value
    end
    index = -1
    if check_input 
        # check in input if input_variables are already there
        # if they are, don't run func (unless redo is true)
        for (i, in_dict) in enumerate(input)
            if in_dict == input_variables
                if !redo
                    run_func = false
                end
                index = i
                break
            end
        end
    end

    if !run_func
        # Load the data
        if printing 
            println("Loading data from file $filename")
        end
        jldopen(filename, "r") do file
            results = file["output"][index]
        end
    else
        if printing 
            println("Constructing data by running function $func")
        end
        predate = Dates.format(now(), "yyyy-mm-dd-HH-MM-SS")
        # Run the function
        results = func(args...; kwargs...)
        postdate = Dates.format(now(), "yyyy-mm-dd-HH-MM-SS")
        # Store inputs in the file
        if exists # file exists
            jldopen(filename, "r+") do file
                if index < 0
                    push!(file["input"], input_variables)
                    push!(file["output"], results)
                    push!(file["predate"], predate)
                    push!(file["postdate"], postdate)
                else
                    file["output"][index] = results
                    file["predate"][index] = predate
                    file["postdate"][index] = postdate
                end
            end
        else
            jldopen(filename, "w") do file
                file["input"] = [input_variables]
                file["output"] = [results]
                file["predate"] = [predate]
                file["postdate"] = [postdate]
            end
        end
    end
    return results
end
## Test 
function fun_name(a, b, c=1; d=1)
    A = a + b + c
    B = a * b * c+ d
    return A, B
end
# run the function using save_io
A, B = save_io(fun_name, 2, 3, d=3, printing=true)

Constructing data by running function fun_name


(6, 9)

In [ ]:
function io_load_inputs(func::Union{Function, String})
    # Load all the input variables for which simulation data has been stored
    filename = joinpath("Results", string(func)*".jld2")
    exists::Bool = isfile(filename)
    # if it does, load it using JLD2
    input::Vector{Dict{String,Any}} = []
    if exists
        # check 
        input = nothing
        jldopen(filename, "r") do file
            # check if input exists in file
            all_keys = keys(file)
            if "input" in all_keys
                # check if args and kwargs exist in file
                check_input = true
                input = file["input"]
            end
        end
    end
    return input
end

# Function idea for quick access to arrays of data from a function_io storage file
#function io_load_data(func::Function, size::Union{Tuple, Vector}, args...; kwargs...)
#    # size specifies the dimensions of the output array, search for the arguments in args or kwargs with those dimensions
#    # these variables do not need to be specified, but if they are, they will be used to filter the data

In [17]:
using SymPy

# Declare symbolic variables
delta = symbols("Delta", real=true)
omega = symbols("omega", real=true)
omega_dash = symbols("Omega", real=true)
kappa = symbols("kappa", real=true)
g = symbols("g", real=true)

g

In [18]:
A = -(4*delta+2im*omega)/((2*delta+im*omega)*(2*delta+im*kappa)-4*g^2)
B = 1/omega_dash*((-4*delta-2im*omega)*(omega-kappa)+16im*g^2)/((2*delta+im*omega)*(2*delta+im*kappa)-4*g^2)

         2                                              
   16*I*g  + (-4*Delta - 2*I*omega)*(-kappa + omega)    
--------------------------------------------------------
      /     2                                          \
Omega*\- 4*g  + (2*Delta + I*kappa)*(2*Delta + I*omega)/

In [23]:
simplify(abs(B)^2/abs(A)^2)

       2      2          2                      2      2       4       2      
4*Delta *kappa  - 8*Delta *kappa*omega + 4*Delta *omega  + 64*g  + 16*g *kappa
------------------------------------------------------------------------------
                                                               2 /       2    
                                                          Omega *\4*Delta  + o

             2      2        2      2                3        4
*omega - 16*g *omega  + kappa *omega  - 2*kappa*omega  + omega 
---------------------------------------------------------------
    2\                                                         
mega /                                                         

In [5]:
# Define your function
f = sin(j*pi*t/T) *exp(-im*delta*(T-t))

# Perform the integral
integral_f = integrate(f, (t,0,T)).doit()

/                                                                       -I*T*d
|  T*sin(T*delta)   I*T*cos(T*delta)   I*sin(T*delta)   cos(T*delta)   e      
|- -------------- - ---------------- - -------------- + ------------ - -------
|        2                 2              2*delta          delta          delt
|                                                                             
|                                                                      -I*T*de
| T*sin(T*delta)   I*T*cos(T*delta)   I*sin(T*delta)   cos(T*delta)   e       
< -------------- + ---------------- + -------------- - ------------ + --------
|       2                 2              2*delta          delta          delta
|                                                                             
|                 j                                                           
|             (-1) *pi*T*j                         pi*T*j                     
|       - -------------------- + -------------------

In [ ]:
using SymPy

# Declare symbolic variables
t = symbols("t", real=true, positive=true)
T = symbols("T", real=true, positive=true)
j = symbols("j", real=true, positive=true, integer=true)
n = symbols("n", real=true, positive=true, integer=true)

# Define your function
f = sin(j*pi*t/T) * (1 - abs(sin(pi*(t-T/2)/T))^n)